# Pilot: Two-Knob Design on UCI Handwritten

Proof-of-concept run before the KITTI instantiation. Not used for any reported result.

Checks, on two views of UCI Handwritten (Fourier, 76-d; profile correlations, 216-d), that:

- **Knob 1** (feature noise) degrades the audited view's own accuracy.
- **Knob 2a** (pair permutation) leaves it unchanged while breaking correspondence.
- Naive early fusion collapses under Knob 2a.

In [1]:
import os
import urllib.request

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
SEED = 0
DATA_URL = (
    "https://raw.githubusercontent.com/mvlearn/mvlearn/main/"
    "mvlearn/datasets/UCImultifeature"
)
DATA_DIR = "mfeat_data"

NOISE_LEVELS = [0.0, 1.0, 3.0, 6.0]  # Knob 1, in feature std units
SWAP_FRACS = [0.0, 0.25, 0.5, 1.0]  # Knob 2a, fraction of pairs permuted
EFFICACY_DROP = 0.2  # minimum Knob 1 accuracy drop for the gate

## Data

In [3]:
def load_view(filename):
    """Load one mfeat view (header row, label in last column)."""
    os.makedirs(DATA_DIR, exist_ok=True)
    path = os.path.join(DATA_DIR, filename)
    if not os.path.exists(path):
        urllib.request.urlretrieve(f"{DATA_URL}/{filename}", path)
    arr = np.genfromtxt(path, delimiter=",")[1:]
    return arr[:, :-1], arr[:, -1].astype(int)


Xa, ya = load_view("mfeat-fou.csv")
Xb, yb = load_view("mfeat-fac.csv")
assert np.array_equal(ya, yb)
y = ya

rng = np.random.default_rng(SEED)
train_idx, test_idx = train_test_split(
    np.arange(len(y)), test_size=0.3, stratify=y, random_state=SEED
)
print(f"view A {Xa.shape}, view B {Xb.shape}")

view A (2000, 76), view B (2000, 216)


In [4]:
def view_accuracy(X_train, y_train, X_test, y_test):
    """Accuracy of a standardized logistic-regression probe on one view."""
    scaler = StandardScaler().fit(X_train)
    clf = LogisticRegression(max_iter=2000).fit(scaler.transform(X_train), y_train)
    return clf.score(scaler.transform(X_test), y_test)


def permute_pairs(idx, frac):
    """Knob 2a: permute a fraction of indices among themselves."""
    idx = idx.copy()
    n = round(frac * len(idx))
    if n > 1:
        chosen = rng.choice(len(idx), n, replace=False)
        idx[chosen] = idx[chosen][rng.permutation(n)]
    return idx

## Knob 1: genuine degradation of view B

In [5]:
feat_std = Xb.std(axis=0)
rows = []
for level in NOISE_LEVELS:
    Xb_noisy = Xb + rng.normal(0, 1, Xb.shape) * feat_std * level
    acc = view_accuracy(
        Xb_noisy[train_idx], y[train_idx], Xb_noisy[test_idx], y[test_idx]
    )
    rows.append({"noise": level, "acc_B": round(acc, 3)})

knob1 = pd.DataFrame(rows).set_index("noise")
knob1

,acc_B
noise,
0.0,0.975
1.0,0.925
3.0,0.640
6.0,0.328


## Knob 2a: semantic misalignment

In [6]:
rows = []
for frac in SWAP_FRACS:
    perm = permute_pairs(test_idx, frac)
    acc = view_accuracy(Xb[train_idx], y[train_idx], Xb[perm], y[perm])
    mismatch = float(np.mean(y[test_idx] != y[perm]))
    rows.append({"frac": frac, "acc_B": round(acc, 3), "mismatch": round(mismatch, 3)})

knob2a = pd.DataFrame(rows).set_index("frac")
knob2a

,acc_B,mismatch
frac,,
0.00,0.975,0.000
0.25,0.975,0.228
0.50,0.975,0.453
1.00,0.975,0.903


## Naive fusion under Knob 2a

In [7]:
scaler_a = StandardScaler().fit(Xa[train_idx])
scaler_b = StandardScaler().fit(Xb[train_idx])


def fuse(idx_a, idx_b):
    """Early fusion: concatenate standardized views A and B."""
    return np.hstack([scaler_a.transform(Xa[idx_a]), scaler_b.transform(Xb[idx_b])])


fusion = LogisticRegression(max_iter=2000).fit(fuse(train_idx, train_idx), y[train_idx])
rows = []
for frac in SWAP_FRACS:
    perm = permute_pairs(test_idx, frac)
    acc = fusion.score(fuse(test_idx, perm), y[test_idx])
    rows.append({"frac": frac, "acc_fused": round(acc, 3)})

fused = pd.DataFrame(rows).set_index("frac")
fused

,acc_fused
frac,
0.00,0.982
0.25,0.753
0.50,0.547
1.00,0.172


## Validity checks

In [8]:
checks = {
    "Knob 1 degrades view B": (
        knob1["acc_B"].iloc[0] - knob1["acc_B"].iloc[-1] > EFFICACY_DROP
    ),
    "Knob 2a leaves view B intact": (
        abs(knob2a["acc_B"].iloc[-1] - knob2a["acc_B"].iloc[0]) < 0.01
    ),
    "Knob 2a breaks correspondence": knob2a["mismatch"].iloc[-1] > 0.8,
    "Knob 2a collapses naive fusion": (
        fused["acc_fused"].iloc[0] - fused["acc_fused"].iloc[-1] > 0.3
    ),
}
for name, ok in checks.items():
    print(f"[{'PASS' if ok else 'FAIL'}] {name}")
assert all(checks.values())

[PASS] Knob 1 degrades view B
[PASS] Knob 2a leaves view B intact
[PASS] Knob 2a breaks correspondence
[PASS] Knob 2a collapses naive fusion
